In [1]:
using Pkg; VERSION, Base.active_project(), Pkg.status()
Pkg.instantiate(); Pkg.precompile()

Status `~/Documents/amoc-edgetrack/Project.toml`
  [336ed68f] CSV v0.10.16
⌃ [608a59af] ChaosTools v3.2.1
  [35d6a980] ColorSchemes v3.31.0
⌅ [a93c6f00] DataFrames v1.7.1
⌃ [5732040d] DelayEmbeddings v2.7.4
⌅ [459566f4] DiffEqCallbacks v2.35.0
⌅ [0c46a032] DifferentialEquations v7.10.0
⌅ [31c24e10] Distributions v0.25.118
⌅ [61744808] DynamicalSystems v3.0.0
⌅ [6e36e845] DynamicalSystemsBase v3.2.2
  [59287772] Formatting v0.4.3
  [f67ccb44] HDF5 v0.17.3
  [5360f785] Haversine v1.1.0
⌅ [7073ff75] IJulia v1.26.0
⌅ [5ab0869b] KernelDensity v0.6.10
  [a5e1c1ea] LatinHypercubeSampling v1.9.0
⌅ [2fda8390] LsqFit v0.15.1
⌅ [1dea7af3] OrdinaryDiffEq v6.58.2
⌅ [91a5bcdd] Plots v1.40.20
⌅ [92933f4c] ProgressMeter v1.10.4
  [90137ffa] StaticArrays v1.9.18
⌅ [2913bbd2] StatsBase v0.33.21
Info Packages marked with ⌃ and ⌅ have new versions available, but those with ⌅ cannot be upgraded. To see why use `status --outdated`


In [62]:
using DynamicalSystemsBase
using DynamicalSystems
using DelayEmbeddings: Dataset
using ChaosTools
using OrdinaryDiffEq
using LinearAlgebra: norm
using Plots
using Statistics
using ProgressMeter
using Random
using Printf
using HDF5
using Dates
using StaticArrays
using DataFrames, CSV

In [51]:
inittest_default(D) = (state1, d0) -> state1 .+ d0/sqrt(D)

"""
    edgetracking(ds, u1, u2, attractors;
        eps1=1e-9, eps2=1e-8, converge=1e-5,
        dt=0.01, ϵ_mapper=0.1, diffeq = (;alg = Vern9()), maxit=100,
        verbose=true, output_level=0, kwargs...
    )

Apply the edge tracking algorithm as described in Skufca et al. (2006) and Lucarini & Bodai
(2017) to the dynamical system `ds` starting from two system states `u1` and `u2` which must
belong to different attractors.

`attractors` is a dictionary that maps a label onto an `N`-dimensional vector, where `N` is
the dimension of `ds`. Chaotic attractors must be approximated by, e.g., their mean.

IMPORTANT: this function is superseded by the `edgetracking` function
           in https://github.com/reykboerner/Attractors.jl/blob/edgetracking/src/boundaries/edgetracking.jl
           Please use this improved implementation for new code.

# Keyword arguments
* `eps1 = 1e-9`: The maximum (Euclidean) distance between two states after bisection,
                 before the start of an edge tracking step
* `eps2 = 1e-8`: The maximum (Euclidean) distance between two states at the end of an edge tracking step
* `converge = 1e-5`: Maximum distance between two successive edge tracking steps that terminates the algorithm.
                     Should not be used (i.e., set to very small values) for chaotic or periodic attractors. Use `maxit` instead.
* `dt = 0.01`: Time interval between two successive steps of the integrator. Not necessarily identical
               to the numerical timestep of the ODE system, see `DynamicalSystems.step!`.
* `ϵ_mapper = 0.01`: passed to `AttractorsViaProximity`
* `diffeq`: tuple of arguments passed to the ODE solver, such as the algorithm and timestepping arguments
* `maxit = 100`: Maximum number of edge tracking steps
* `verbose = true`: Print diagnostic output at every iteration
* `output_level = 0`: Switch between different outputs.
                      0 = only return final state vector
                      1 = return edge and bracketing trajectory after every iteration
                      2 = return edge and bracketing trajectory every Δt time units
* `kwargs...`: Keyword arguments passed on to `AttractorsViaProximity`
"""
function edgetracking(ds::DynamicalSystem, u1, u2, attractors::Dict;
    eps1=1e-9,
    eps2=1e-8,
    converge=1e-5,
    dt=0.01,
    ϵ_mapper=0.1,
    #diffeq = (;alg = Vern9()),
    maxit=100,
    verbose=true,
    output_level=0,
    kwargs...)
    
    pinteg = parallel_integrator(ds, [u1, u2])
    mapper = AttractorsViaProximity(ds, attractors, ϵ_mapper; kwargs...)

    track_edge(pinteg, mapper;
        eps1=eps1, eps2=eps2, converge=converge, dt=dt, maxit=maxit,
        verbose=verbose, calc_lyapunovs=false, output_level=output_level)
end;


"""
    track_edge(pinteg, mapper;
        eps1=1e-9, eps2=1e-8, converge=1e-5, dt=0.01, maxit=100,
        verbose=false, output_level=0,
        calc_lyapunovs=false, lyapunov_spinup=200, d0=1e-9, lower_threshold=1e-11, upper_threshold=1e-7
    )

Main function for the edge tracking algorithm (Battelino et al. 1988, Skufca et al. 2006).

This function is called from `edgetracking`, see its documentation for an explanation of parameters.

Note: `calc_lyapunovs` and following keyword arguments are experimental,
      are NOT scientifically validated
      and do NOT reproduce the results from Mehling et al. 2023, arXiv:2308.16251
"""
function track_edge(pinteg, mapper::AttractorMapper;
    eps1=1e-9, eps2=1e-8, converge=1e-5, dt=0.01, maxit=100,
    verbose=false, output_level=0,
    calc_lyapunovs=false, lyapunov_spinup=200, d0=1e-9, lower_threshold=1e-11, upper_threshold=1e-7)
    
    lyapunov_spinup += 1
    verbose && println("=== Starting edge tracking algorithm ===")
    
    u1, u2 = bisect_to_edge(pinteg, mapper; abstol=eps1)
    edgestate = (u1 + u2)/2
    
    if output_level>=1
        track1 = [u1]
        track2 = [u2]
        edge = [edgestate]
    end
    # Initialize Lyapunov "test" trajectories
    pinteg_test = deepcopy(pinteg)
    inittest = inittest_default(size(get_state(pinteg, 1))[1])
    λ1 = zero(d0)
    λ2 = zero(d0)
    titer = zero(d0)
    # End Lyapunov init

    verbose && println("... Iteration 1: Edge at $(edgestate)")

    correction = converge + 1
    counter = 1
    while (correction > converge) & (maxit > counter)
        # Initialize/rescale Lyapunov calculation
        if calc_lyapunovs
            if counter == lyapunov_spinup
                println("Initializing Lyapunov")
                reinit!(pinteg_test, [inittest(u1, d0), inittest(u2, d0)])
                d_test1 = deepcopy(d0)
                d_test2 = deepcopy(d0)
            elseif counter > lyapunov_spinup
                # get distance between test and shadowing trajectories before rescaling the latter
                d_test1 = norm(get_state(pinteg_test, 1) - get_state(pinteg, 1))
                d_test2 = norm(get_state(pinteg_test, 2) - get_state(pinteg, 2))
                println("Re-initializing Lyapunov, distances:", d_test1, " ", d_test2)
                # re-adjust (but not re-scale) trajectory
                #reinit!(pinteg_test, [inittest(u1, d_test1), inittest(u2, d_test2)]) #OLD
                reinit!(pinteg_test, [
                    u1 + (pinteg_test.u[1] - pinteg.u[1])/(d_test1/d0),
                    u2 + (pinteg_test.u[2] - pinteg.u[2])/(d_test2/d0)
                ])
            end
        end
        # Re-initalize shawding trajectories
        reinit!(pinteg, [u1,u2])
        state = edgestate
        distance = norm(get_state(pinteg, 1)-get_state(pinteg, 2))
        while distance < eps2
            step!(pinteg, dt)
            if calc_lyapunovs && counter>=lyapunov_spinup
                step!(pinteg_test, dt)
                d_test1 = norm(get_state(pinteg_test, 1) - get_state(pinteg, 1))
                d_test2 = norm(get_state(pinteg_test, 2) - get_state(pinteg, 2))
                #println(titer, " ", d_test1, " ", d_test2)
                if (d_test1 < lower_threshold) || (d_test1 > upper_threshold)
                    # "threshold" rescale
                    a1 = d_test1/d0
                    #println("Rescaling state 1: a1 = ", a1)
                    λ1 += log(a1)
                    #pinteg_test.u[1] = inittest(get_state(pinteg, 1), d0)
                    pinteg_test.u[1] = pinteg.u[1] + (pinteg_test.u[1] - pinteg.u[1])/a1
                    u_modified!(pinteg_test, true)
                    d_test1 = norm(get_state(pinteg_test, 1) - get_state(pinteg, 1))
                    #println(d_test1)
                end
                if (d_test2 < lower_threshold) || (d_test2 > upper_threshold)
                    a2 = d_test2/d0
                    #println("Rescaling state 2: a2 = ", a2)
                    λ2 += log(a2)
                    #pinteg_test.u[2] = inittest(get_state(pinteg, 2), d0)
                    pinteg_test.u[2] = pinteg.u[2] + (pinteg_test.u[2] - pinteg.u[2])/a2
                    u_modified!(pinteg_test, true)
                    d_test2 = norm(get_state(pinteg_test, 2) - get_state(pinteg, 2))
                    #println(d_test2)
                end
                titer += 1
            end
            state1 = get_state(pinteg, 1)
            state2 = get_state(pinteg, 2)
            distance = norm(state1-state2)
            if output_level>=2
                push!(track1, state1)
                push!(track2, state2)
                push!(edge, (state1+state2)/2)
            end
        end
        u1, u2 = bisect_to_edge(pinteg, mapper; abstol=eps1)
        edgestate = (u1 + u2)/2
        correction = norm(edgestate - state)
        counter += 1

        if output_level==1
            push!(track1, u1)
            push!(track2, u2)
            push!(edge, edgestate)
        end
        
        verbose && println("... Iteration $(counter): Edge at $(edgestate)")
        (counter == maxit) && @warn("Reached maximum number of $(maxit) iterations; did not converge.")
    end

    (counter < maxit) && println("Edge-tracking converged after $(counter) iterations.")

    if calc_lyapunovs
        # Do finale rescale
        d_test1 = norm(get_state(pinteg_test, 1) - get_state(pinteg, 1))
        d_test2 = norm(get_state(pinteg_test, 2) - get_state(pinteg, 2))
        # store growth rates
        a1 = d_test1/d0
        a2 = d_test2/d0
        λ1 += log(a1)
        λ2 += log(a2)
        println("Lyapunov exponents:\n", λ1/(titer*dt), " ", λ2/(titer*dt), "(iters = ", titer, ")")
    end

    if output_level>=1
        return Dataset(edge), Dataset(track1), Dataset(track2)
    else
        return edgestate
    end
end;

"""
    bisect_to_edge(pinteg, mapper; abstol=1e-9)

Find the basin boundary between two states `u1, u2 = get_states(pinteg)` by bisecting along a
straight line in phase space. The states `u1` and `u2` must belong to different basins.
Returns two new states located on either side of the basin boundary at a maximum distance of
`abstol` between each other.

`pinteg` is a `parallel_integrator` with two states. The `mapper` must be an `AttractorMapper`
of subtype `AttractorsViaProximity` or `AttractorsViaRecurrences`.

Note: If the straight line between `u1` and `u2` intersects the basin boundary multiple
times, the method will find one of these intersection points. If more than two attractors
exist, one of the two returned states may belong to a different basin than the initial
conditions `u1` and `u2`. A warning is raised if the bisection involves a third basin.

# Keyword arguments
* `abstol = 1e-9`: The maximum (Euclidean) distance between the two returned states.
"""
function bisect_to_edge(pinteg, mapper::AttractorMapper; abstol=1e-9)
    u1 = collect(get_state(pinteg,1))
    u2 = collect(get_state(pinteg,2))
    idx1, idx2 = mapper(u1), mapper(u2)
    
    if idx1 == idx2
        error("Both initial conditions belong to the same basin of attraction: ", idx1)
    end
    
    distance = norm(u1-u2)
    while distance > abstol
        u_new = (u1 + u2)/2
        idx_new = mapper(u_new)
        # todo: what if u_new lies on boundary
        
        if idx_new == idx1
            u1 = u_new
        else 
            u2 = u_new
            if idx_new != idx2
                println("Warning: New bisection point belongs to a third basin of attraction or diverges: ", idx_new) # @warn
            end
        end    
        distance = norm(u1-u2)
    end
    [u1, u2]
end;


In [52]:


# Smooth absolute value function
function smoothabs(x, xi=10000)
    x*tanh(x*xi)
end

# L84 model equations
function L84(u,p,t)
    x, y, z = u
    a, b, F, G = p[1]

    Δ = y^2 + z^2
    dx = -Δ - a*(x - F)
    dy = x*y - b*x*z - (y - G)
    dz = b*x*y + x*z - z

    SA[dx, dy, dz]
end

# Stommel model equations following the Gottwald (2021) formulation
function stommel_gottwald(u,p,t)
    T, S = u
    μ, ϵa, θ0, σ0 = p[1]

    T_surf = θ0
    S_surf = σ0
    dT = -1/ϵa*(T - T_surf) - T - μ*smoothabs(S-T)*T
    dS = S_surf - S - μ*smoothabs(S-T)*S

    SA[dT, dS]
end

# Gottwald model functions without sea ice
function gottwald_noice(t,u,p; stochsys=true)
    x, y, z, T, S = u
    if stochsys
        pvec = (isa(p, AbstractVector) && isa(p[1], AbstractVector)) ? p[1] : p
    else
        pvec = p
    end
    a, b, μ, ϵa, ϵf, F0, F1, G0, G1, θ0, θ1, σ0, σ1, x_mean, Δ_mean = pvec

    Δ = y^2 + z^2
    T_surf = θ0 + θ1 * (x - x_mean)/(sqrt(ϵf))
    S_surf = σ0 + σ1 * (Δ - Δ_mean)/(sqrt(ϵf))
    dx = 1/ϵf*(-Δ - a*(x - F0 - F1*T))
    dy = 1/ϵf*(x*y - b*x*z - (y - G0 + G1*T))
    dz = 1/ϵf*(b*x*y + x*z - z)
    dT = -1/ϵa*(T - T_surf) - T - μ*smoothabs(S-T)*T
    dS = S_surf - S - μ*smoothabs(S-T)*S

    SA[dx, dy, dz, dT, dS]
end

# Rescaled (x/y/z) for edge tracking
function gottwald_noice_scaled(u,p,t; stochsys=true)
    x, y, z, T, S = u
    if stochsys
        pvec = p[1]
    else
        pvec = p
    end
    a, b, μ, ϵa, ϵf, F0, F1, G0, G1, θ0, θ1, σ0, σ1, x_mean, Δ_mean, s = pvec

    Δ = y^2 + z^2
    T_surf = θ0 + θ1 * (s*x - x_mean)/(sqrt(ϵf))
    S_surf = σ0 + σ1 * (s^2*Δ - Δ_mean)/(sqrt(ϵf))
    dx = 1/ϵf*(-s*Δ - a/s*(s*x - F0 - F1*T))
    dy = 1/ϵf*(s*x*y - s*b*x*z - (s*y - G0 + G1*T)/s)
    dz = 1/ϵf*(s*b*x*y + s*x*z - z)
    dT = -1/ϵa*(T - T_surf) - T - μ*smoothabs(S-T)*T
    dS = S_surf - S - μ*smoothabs(S-T)*S

    SA[dx, dy, dz, dT, dS]
end

# Gottwald model functions without sea ice (inplace)
function gottwald_noice!(du,u,p,t)
    x, y, z, T, S = u
    a, b, μ, ϵa, ϵf, F0, F1, G0, G1, θ0, θ1, σ0, σ1, x_mean, Δ_mean = p[1]

    Δ = y^2 + z^2
    T_surf = θ0 + θ1 * (x - x_mean)/(sqrt(ϵf))
    S_surf = σ0 + σ1 * (Δ - Δ_mean)/(sqrt(ϵf))
    du[1] = 1/ϵf*(-Δ - a*(x - F0 - F1*T)) # x
    du[2] = 1/ϵf*(x*y - b*x*z - (y - G0 + G1*T)) # y
    du[3] = 1/ϵf*(b*x*y + x*z - z) # z
    du[4] = -1/ϵa*(T - T_surf) - T - μ*smoothabs(S-T)*T # T
    du[5] = S_surf - S - μ*smoothabs(S-T)*S # S
end

# Gottwald model functions without sea ice, but with seasonal cycle
function gottwald_noice_seasons(u,p,t)
    x, y, z, T, S = u
    a, b, μ, ϵa, ϵf, F0, F1, G0, G1, θ0, θ1, σ0, σ1, x_mean, Δ_mean, td, F2, G2 = p[1]

    ω = 2.0*pi*td # annual frequency
    Δ = y^2 + z^2
    T_surf = θ0 + θ1 * (x - x_mean)/(sqrt(ϵf))
    S_surf = σ0 + σ1 * (Δ - Δ_mean)/(sqrt(ϵf))
    dx = 1/ϵf*(-Δ - a*(x - F0 - F1*T - F2*cos(ω * t)))
    dy = 1/ϵf*(x*y - b*x*z - (y - G0 + G1*T - G2*cos(ω * t)))
    dz = 1/ϵf*(b*x*y + x*z - z)
    dT = -1/ϵa*(T - T_surf) - T - μ*smoothabs(S-T)*T
    dS = S_surf - S - μ*smoothabs(S-T)*S

    SA[dx, dy, dz, dT, dS]
end

# Default parameter values
function param_gwn_default(param_l84=nothing,σ0=0.9)
    if !isnothing(param_l84)
        # Long run of L84 model to determine mean and std of coupling quantities
        model_l84 = StochSystem(L84, param_l84, 3, 0.016)
        sim_l84 = relax(model_l84, [-1.5, 1.5, 1.5], tspan=(0.,5000.), solver=Tsit5(), reltol=1e-10, abstol=1e-10, saveat=range(0,5000,500001))
        x_mean = mean(sim_l84[1,50000:end])
        Δ_mean = mean(sim_l84[2,50000:end].^2 + sim_l84[3,50000:end].^2)
        x_std = std(sim_l84[1,50000:end])
        Δ_std = std(sim_l84[2,50000:end].^2 + sim_l84[3,50000:end].^2)
    else
        param_l84 = [0.25, 4., 8., 1.] # default L84 params [a, b, F0, G0]
        x_std = 0.513; Δ_std = 1.071; x_mean = 1.0147; Δ_mean = 1.7463 # long-run results from default L84
    end
    # Unpack L84 params
    a, b, F0, G0 = param_l84
    # Other default model params
    μ = 7.5; θ0 = 1.
    #σ0 = 0.9 # Stommel params
    ϵa = 0.34; ϵf = .0003 # timescales
    F1 = 0.1; G1 = 0.; # coupling params
    perturb_scaling = 0.01 # coupling strength of L84 to Stommel

    # Coupling params from L84 run/default
    θ1 = 0.0195
    #min(θ0, perturb_scaling/x_std)
    σ1 = 0.00934
    #min(σ0, perturb_scaling/Δ_std)

    [a, b, μ, ϵa, ϵf, F0, F1, G0, G1, θ0, θ1, σ0, σ1, x_mean, Δ_mean]
end

param_gwn_default (generic function with 3 methods)

In [53]:
write_output = true

function output_matrix(dataset; dtype=Float32)
    out_matrix = zeros(dtype, size(dataset))
    for (i, state) in enumerate(dataset)
        out_matrix[i,:] = convert(Array{dtype}, collect(state))
    end
    return out_matrix
end

# G21 model with scaling of (x,y,z)
sys2(u,p,t) = gottwald_noice(t,u,p;stochsys=true)
scale = 5000.
# Parameters


sigma_0 = try
    parse(Float64, ARGS[1])
catch
    0.9
end
p_default = param_gwn_default()
p_scaled = zeros(length(p_default)+1)
p_scaled[1:end-1]=p_default[:]
p_scaled[7]=0.1 # F1
p_scaled[5]=3e-4 # epsilon_f
p_scaled[12]=sigma_0  # freshwater flux, cmd line argument
p_scaled[end]=scale
# Initialize ds
u1=[2.,1.,0.,1.5,0.5]./[scale, scale, scale, 1., 1.]
u2=[1., 0., 2., 1.5, 1.125]./[scale, scale, scale, 1., 1.]
ds = CoupledODEs(sys2, u1, p_scaled; diffeq=(alg = DP5(), adaptive = false, dt = 7.5e-6))

# Timestepping arguments
diffeq_args = (alg = DP5(), adaptive=false, dt=7.5e-6)
t_tot = 100. # total time in time units (for attractor trajectories)
spinup = 150. # spinup time in time units (-"-)
Δt = diffeq_args.dt*10
tsol = 0:Δt:t_tot

# Find attractors by forward simulation
sol_u1 = trajectory(ds, t_tot, u1; Ttr=spinup, Δt=Δt)
states1, t1 = sol_u1
u_mat1 = output_matrix(states1)
mean1 = vec(mean(u_mat1, dims=1))   # 1×N -> N-vector

sol_u2 = trajectory(ds, t_tot, u2; Ttr=spinup, Δt=Δt)
states2, t2 = sol_u2
u_mat2 = output_matrix(states2)
mean2 = vec(mean(u_mat2, dims=1))
  attrs = Dict(
    1 => StateSpaceSet(states1),   # use the trajectory states directly
    2 => StateSpaceSet(states2)
  )
#attrs = Dict(1 =>Dataset([mean(sol_u1.data)]), 2 => Dataset([mean(sol_u2.data)]))

lya_a1 = lyapunov(ds, t_tot; u0=u1, Ttr=spinup, Δt=Δt,
                  d0=1e-9, d0_upper=1e-7, d0_lower=1e-11)
lya_a2 = lyapunov(ds, t_tot; u0=u2, Ttr=spinup, Δt=Δt,
                  d0=1e-9, d0_upper=1e-7, d0_lower=1e-11)
#lya_a1 = lyapunov(ds, t_tot; u0 = u1, Ttr=spinup, upper_threshold=1e-7, lower_threshold=1e-11, Δt=Δt)
#lya_a2 = lyapunov(ds, t_tot; u0 = u2, Ttr=spinup, upper_threshold=1e-7, lower_threshold=1e-11, Δt=Δt)

println("Attractor 1: ", Matrix(attrs[1]))
println("(Lyapunov = ", lya_a1, ")")
println("Attractor 2: ", Matrix(attrs[2]))
println("(Lyapunov = ", lya_a2, ")")

t1 = now()

Niter = 2000 # iterations for edge tracking algorithm
edge, track1, track2 = edgetracking(
                        ds, u1, u2, attrs,
                        eps1 = 0.0025, eps2 = 0.004, dt=7.5e-6, ϵ_mapper=0.01,
                        maxit=Niter,
                        output_level=2
                        )

tedge = 0:7.5e-6:(size(edge)[1]-1)*7.5e-6

# Write output to HDF5 file
if write_output
    h5open(@sprintf("edgetrack_L84stommel_sig%04d.h5", sigma_0*1000), "w") do file
        write(file, "t", convert(Array{Float32}, tedge))
        write(file, "u_edge", output_matrix(edge))
        write(file, "u_track1", output_matrix(track1))
        write(file, "u_track2", output_matrix(track2))
    end
end

t2 = now()
print("Edge tracking finished successfully. Wall-clock time: ")
println(canonicalize(t2 - t1))


Attractor 1: [1.7632018313626319 0.6632185831547485 -0.18522868709924942 0.5655235716763684 0.39110632406440626; 1.9510485385222192 0.10771260701745267 1.0576575466179365 0.5657343962235198 0.3910676104531729; 2.0009875836331146 -1.172758489692616 -0.20771660634748262 0.5659723095079429 0.3910500458406893; 1.9717909461161485 0.9888645718232996 -1.0522046578416486 0.5662143984098825 0.3910450526700884; 1.6004785163523172 1.0894312110824091 1.692490572427882 0.5664133900380768 0.3910971040447006; 0.8949086609177508 -1.1529971556398784 1.793431415587104 0.5664701953684633 0.39120898991201286; 0.38944426643041247 -1.5624026071956305 0.7962846888190179 0.5663665469935615 0.39129504970164397; 0.30210522714983373 -1.2331209153308613 0.2642800031302357 0.5661900613323714 0.3913164734084816; 0.5042216728183914 -0.8387179164612303 -0.14130553513809904 0.566031578442073 0.39129038070371425; 0.8419574175961808 -0.3038079396179715 -0.4965355084709398 0.5659428632802853 0.3912395369020875; 1.2008849

Excessive output truncated after 524302 bytes.

 0.3973868139436909; 1.5242848325515774 0.682478309979346 -1.3169815706639678 0.5667918731632017 0.3973947637842858; 1.2372669133228598 1.7468629450934987 0.6607927144599454 0.5668882875951488 0.3974376223509378; 0.7024110660138217 0.6374161129900823 1.9105542180628565 0.5668784540204092 0.3975248924532848; 0.253205762949099 -0.010515397160935101 1.7846881205113827 0.5667389674244946 0.3976036101826365; 0.09616640482536343 -0.0008095132694102618 1.438994122099197 0.5665235648729694 0.39763820394032007; 0.17280304078050324 0.0852574335828202 1.160901986522401 0.5663007703146218 0.3976353533474687; 0.37388161020434296 0.04027420276737094 0.9842538297186122 0.5661144804080046 0.39761093623904253; 0.6346610328465848 -0.16032120224827237 0.8408697172168456 0.5659864746602808 0.39757452596315834; 0.9335248952471463 -0.44957425630109 0.5540972484531999 0.5659281288663256 0.39752913946876106; 1.2721132759040963 -0.5128101966309774 -0.019688606996116905 0.5659488526957654 0.3974741090376779; 1.

In [67]:
#plotting edge state T,S

#get numeric arrays
edge_mat = output_matrix(edge)          
t_edge = convert(Array{Float64}, tedge) 

#plot T and S
p=plot(t_edge, edge_mat[:,4], label="T (edge)", xlabel="time", ylabel="value",
         title="Edge state in phase space of T and S")
plot!(p,t_edge, edge_mat[:,5], label="S (edge)")
savefig(p, "edge_state_TS.png")

julia(23664) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


"/Users/amethyst/Documents/amoc-edgetrack/edge_state_TS.png"

In [57]:
#=
Look at dynamics of edge state
final values of T, and S
maximum change in edge state to check convergence
=#

edge_mat = output_matrix(edge)          
dt = 7.5e-6
t_edge = collect(0:dt:(size(edge)[1]-1)*dt)   
total_time = last(t_edge) 
final_state = edge_mat[end, :]        
T_final = final_state[4]
print(T_final)
S_final = final_state[5]
print(S_final)
T_series = edge_mat[:, 4]
S_series = edge_mat[:, 5]
first_T, last_T = T_series[1], T_series[end]
first_S, last_S = S_series[1], S_series[end]
k = min(50, size(edge_mat,1)-1)
max_change = maximum([norm(edge_mat[end-i,:] .- edge_mat[end-i-1,:]) for i in 0:(k-1)])

0.68356270.61453646Float32[1.9642208 -1.8406317 0.5506342 0.6834161 0.6144572; 1.9092389 -1.9323802 0.1929057 0.68343824 0.614465; 1.8527298 -1.9523219 -0.17331819 0.6834589 0.61447304; 1.7949183 -1.902687 -0.5331855 0.68347824 0.61448145; 1.7359654 -1.7887388 -0.87346286 0.6834961 0.61449003; 1.6759702 -1.6181281 -1.1830691 0.68351245 0.61449885; 1.6149769 -1.4001971 -1.4533988 0.68352735 0.6145079; 1.5529845 -1.1452898 -1.6784527 0.6835407 0.6145172; 1.4899586 -0.86411005 -1.8548028 0.68355244 0.6145267; 1.4258454 -0.56715965 -1.9814233 0.6835627 0.61453646]


In [60]:
#=
Look at max change in edge state.
0.375 is large since the convergence tolerance of the edge state was 1e-5.
However, based on the std of T and S, the edge state is not converging for x,y,and z.
=#
M = min(200, size(edge_mat,1)-1)
println("max change last $M = ", maximum([norm(edge_mat[end-i,:] .- edge_mat[end-i-1,:]) for i in 0:(M-1)]))
println("std T last $M = ", std(edge_mat[end-M+1:end,4]))
println("std S last $M = ", std(edge_mat[end-M+1:end,5]))

max change last 200 = 0.37490177
std T last 200 = 0.00041549123
std S last 200 = 0.00011937455


In [61]:
#=
pick a window size to see mean and median of recent T,S in edge state
=#
M = min(200, size(edge_mat,1))
tail = edge_mat[end-M+1:end, :]
T_mean = mean(tail[:,4]); T_std = std(tail[:,4])
S_mean = mean(tail[:,5]); S_std = std(tail[:,5])
T_median = median(tail[:,4]); S_median = median(tail[:,5])

println("T_mean ± std = ", T_mean, " ± ", T_std)
println("S_mean ± std = ", S_mean, " ± ", S_std)
println("T_median = ", T_median, ", S_median = ", S_median)

T_mean ± std = 0.682514 ± 0.00041549123
S_mean ± std = 0.6146085 ± 0.00011937455
T_median = 0.6824541, S_median = 0.6145892


In [64]:
edge_mat = output_matrix(edge)            
dt = 7.5e-6
t_edge = collect(0:dt:(size(edge_mat,1)-1)*dt)

df = DataFrame(time = t_edge,
               x = Float64.(edge_mat[:,1]),
               y = Float64.(edge_mat[:,2]),
               z = Float64.(edge_mat[:,3]),
               T = Float64.(edge_mat[:,4]),
               S = Float64.(edge_mat[:,5]))

CSV.write("edge_states.csv", df)

"edge_states.csv"